# FSL-AKE-IoD — Proposed Protocol (v2, after independent review)

$\text{Forward-Secure, Lightweight, desynchronisation-resilient AKE between drone }DE_i\text{ and ground station }ES_j$

This notebook describes the protocol **as it stands after the review** (`proposed/code/fslake.py`).
The message formulas are the ones in the Week 3 draft and in `proposed_fslake.py` (v1).
The review found flaws in v1 and fixed them. Each fix is marked **F1–F4** below; the attacks that motivate them are in
`FSL-AKE-IoD-security_verification.ipynb`.

| | v1 (draft) | v2 (this notebook) |
|---|---|---|
| F1 | `enroll()` gives every drone `dev="DE"`, so one drone's session deletes all the others | unique device id per drone |
| F2 | replay-cache entry kept until *receive time* + $\Delta T$ | kept until $T_1+\Delta T$ (the whole acceptance window) |
| F3 | exactly one *old* + one *cur* record | *anchor* (last proven key) + *candidates*; promote on proof of possession |
| F4 | drone may re-send MSG1 at any time | drone waits $>2\Delta T$ before re-sending on the same credentials |
| — | no reboot handling | ES refuses MSG1 for $2\Delta T$ after losing its cache; collision-tolerant TID lookup |

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "proposed", "code"))
import pandas as pd
pd.set_option("display.max_colwidth", 200)
from common import Clock, AuthError, h, xor, size_bits, reset_hash_counter, hashes
import fslake as F
print(F.V2)

Config(M1_key_evolution=True, M2_tid_update=True, M3_dual_record=True, M4_replay_cache=True, M5_masked_verifier=True, M6_two_message=True, F2_cache_expiry_from_T1=True, F3_candidate_set=True, F4_retry_gap=True, retry_pseudonyms=0, tid_bytes=20, delta_t=2)


**Output explanation.** `F.V2` is the full protocol: the six design modifications M1–M6 of the draft and the four
review fixes F2–F4 are all switched on. The ablation notebook switches them off one at a time.

## Notation

$$
\begin{aligned}
K_i&\colon\;\text{current 256-bit key shared by }DE_i\text{ and }ES_j\text{ (evolves every session)}\\
TID_i&\colon\;\text{160-bit one-time pseudonym (evolves every session, never sent in advance)}\\
X_{ES}&\colon\;\text{256-bit master secret of }ES_j\text{, kept in a secure element (TPM/HSM)}\\
C_i&=K_i\oplus h(X_{ES}\parallel TID_i)\colon\;\text{masked verifier stored in the ES database}\\
N_1,N_2&\colon\;\text{160-bit nonces (public)}\qquad T_1,T_2\colon\;\text{32-bit timestamps}\\
\Delta T&\colon\;\text{maximum transmission delay}\qquad h(\cdot)\colon\;\text{SHA-256 over a length-prefixed concatenation}
\end{aligned}
$$

## Registration (RA, offline, secure channel)

* $RA$ picks $K_i$ (256 bit), $TID_i$ (160 bit) and an internal device id $dev_i$ (**F1**).
* $DE_i$ stores $\{TID_i, K_i\}$ (52 bytes).
* $ES_j$ stores the *anchor* record $\{dev_i, TID_i, C_i = K_i\oplus h(X_{ES}\parallel TID_i)\}$ and discards $K_i$.
* The RA must erase $K_i$ after provisioning (otherwise it is a privileged insider — see the security notebook).

In [2]:
ra = F.RA(); clock = Clock(); es = F.GroundStation(clock)
creds = [ra.register_drone() for _ in range(2)]
for c in creds:
    es.enroll(c)
drones = [F.Drone(c, clock) for c in creds]
print("drone memory:", {k: (v.hex()[:16] + '...', len(v)) for k, v in drones[0].mem.items()})
print("ES records  :")
for dev, d in es.devs.items():
    print(f"  {dev}: anchor TID={d['anchor']['TID'].hex()[:12]}..  C={d['anchor']['C'].hex()[:16]}..  candidates={len(d['cands'])}")
print("C == K ?", es.devs[creds[0]['dev']]['anchor']['C'] == creds[0]['K'])

drone memory: {'TID': ('3fd8f443e50b4ecb...', 20), 'K': ('d872e65fa72cbc5e...', 32)}
ES records  :
  dev1-d865b739: anchor TID=3fd8f443e50b..  C=bb7f3cd519b8bdfd..  candidates=0
  dev2-f9fb9237: anchor TID=768110fd966a..  C=8d4846c47e56d946..  candidates=0
C == K ? False


**Output explanation.** Each drone holds only its pseudonym and key (20 + 32 bytes). The ES stores one anchor record
per drone, keyed by a unique device id. The stored value `C` differs from `K`: without `X_ES` it is a uniformly random
string.

## Authentication and key establishment (2 messages)

**Step 1 — $DE_i\rightarrow ES_j$.** Pick $N_1, T_1$; $V_1 = h(K\parallel TID\parallel N_1\parallel T_1)$;
$MSG_1=\{TID, N_1, T_1, V_1\}$ (160+160+32+256 = 608 bits).

**Step 2 — $ES_j$.**
* check $|T_1-T_1^*|\le\Delta T$ and $(TID,T_1,N_1)\notin$ cache (entry kept until $T_1+\Delta T$, **F2**)
* for each record stored under $TID$: $K = C\oplus h(X_{ES}\parallel TID)$, check $V_1$
* pick $N_2,T_2$; $SK=h(K\parallel N_1\parallel N_2\parallel T_1\parallel T_2\parallel TID)$; $V_2=h(SK\parallel N_2\parallel T_2)$
* $MSG_2=\{N_2,T_2,V_2\}$ (160+32+256 = 448 bits)
* $K'=h(K\parallel SK)$, $TID'=h(TID\parallel SK)_{[0..159]}$, store $C'=K'\oplus h(X_{ES}\parallel TID')$:
  if the matched record is the **anchor**, add $(TID',C')$ as a **candidate**; if it is a **candidate**, the drone has
  proved it holds it, so promote it to anchor and drop everything else (**F3**).

**Step 3 — $DE_i$.** Check $|T_2-T_2^*|\le\Delta T$, recompute $SK$, check $V_2$; then
$K\leftarrow h(K\parallel SK)$, $TID\leftarrow h(TID\parallel SK)$, erase $N_1$ and the old $K$.

In [3]:
de = drones[0]
reset_hash_counter(); m1 = de.start(); h_de1 = hashes(); clock.advance()
reset_hash_counter(); m2, sk_es = es.respond(m1); h_es = hashes(); clock.advance()
reset_hash_counter(); sk_de = de.finish(m2); h_de2 = hashes()
print("MSG1:", {k: (v if isinstance(v, int) else v.hex()[:12] + '..') for k, v in m1.items()}, size_bits(m1), "bits")
print("MSG2:", {k: (v if isinstance(v, int) else v.hex()[:12] + '..') for k, v in m2.items()}, size_bits(m2), "bits")
print("hashes: DE", h_de1 + h_de2, " ES", h_es)
print("SK agree:", sk_de == sk_es, "  drone TID changed:", de.TID != m1['TID'])
d = es.devs[creds[0]['dev']]
print("ES after session: anchor TID == old TID:", d['anchor']['TID'] == m1['TID'],
      "| candidate TID == drone's new TID:", d['cands'][0]['TID'] == de.TID)

MSG1: {'TID': '3fd8f443e50b..', 'N1': 'a1d3d0335fa4..', 'T1': 1700000000, 'V1': '6ea6c3d7fa8d..'} 608 bits
MSG2: {'N2': '097c5ea0ea75..', 'T2': 1700000001, 'V2': 'df24ed21f381..'} 448 bits
hashes: DE 5  ES 7
SK agree: True   drone TID changed: True
ES after session: anchor TID == old TID: True | candidate TID == drone's new TID: True


**Output explanation.** The two messages are 608 + 448 = 1056 bits. Instrumentation counts 5 hash operations on the
drone (V1, SK, V2, K', TID') and 7 on the ES (unmask, V1, SK, V2, K', TID', mask). Both sides derive the same SK.
The drone's next pseudonym never appears on the air. The ES keeps the old key as the anchor until the drone proves,
in its next session, that it holds the candidate.

## Resynchronisation (lost MSG2) and the retry rule F4

If $MSG_2$ is lost, the drone still holds the anchor credentials. Its next $MSG_1$ matches the anchor, and the ES adds
another candidate; nothing is deleted, so no ordering of deliveries can remove the record the drone actually holds.
The drone may only re-send on the same credentials after $2\Delta T$ (**F4**). Then at most one of its $MSG_1$ is
fresh at the ES at any time, so the bounded candidate list can never be flooded with withheld retries.

In [4]:
de = drones[1]
tr, a, b = F.run_session(de, es, clock, drop="MSG2")
d = es.devs[creds[1]['dev']]
print("after dropped MSG2 -> drone still on anchor:", de.TID == d['anchor']['TID'], "| candidates:", len(d['cands']))
try:
    de.start()
except AuthError as e:
    print("immediate retry:", e)
clock.advance(2 * F.DELTA_T + 1)
tr, a, b = F.run_session(de, es, clock)
print("retry after 2dT: keys agree =", a == b, "| candidates now:", len(d['cands']))
clock.advance(10)
tr, a, b = F.run_session(de, es, clock)
print("next session (drone proves candidate -> promoted): keys agree =", a == b,
      "| anchor == TID the drone just used:", d['anchor']['TID'] == tr['MSG1']['TID'], "| candidates:", len(d['cands']))

after dropped MSG2 -> drone still on anchor: True | candidates: 1
immediate retry: DE: retry on the same credentials before 2*dT has passed (rule F4)
retry after 2dT: keys agree = True | candidates now: 2
next session (drone proves candidate -> promoted): keys agree = True | anchor == TID the drone just used: True | candidates: 1


**Output explanation.** After the drop, the drone is still on the anchor. An immediate retry is refused by rule F4.
The retry after 2ΔT succeeds, and the ES now holds two candidates. When the drone next authenticates with the one it
accepted, that candidate becomes the anchor and the stale candidate is deleted. No extra message is needed.

## Replay cache, reboot and revocation

* The cache stores $(TID,T_1,N_1)$ only **after** a valid $V_1$, so forged messages cannot fill it.
* After a reboot (cache lost) the ES refuses every $MSG_1$ for $2\Delta T$, so every message it might have cached is stale.
* Revocation deletes the anchor **and** all candidates of the device.

In [5]:
de = drones[0]
m1 = de.start(); m2, _ = es.respond(m1); de.finish(m2)
try: es.respond(dict(m1))
except AuthError as e: print("replay:", e)
es.reboot()
try: es.respond(dict(m1))
except AuthError as e: print("replay right after reboot:", e)
clock.advance(2 * F.DELTA_T + 1)
try: es.respond(dict(m1))
except AuthError as e: print("replay after the quiet period:", e)
es.revoke(creds[0]['dev'])
try: F.run_session(de, es, clock)
except AuthError as e: print("revoked drone:", e)

replay: ES: replay detected (cache)
replay right after reboot: ES: quiet period after reboot
replay after the quiet period: ES: stale T1
revoked drone: ES: unknown TID


**Output explanation.** The replay is caught by the cache. Right after a reboot it is caught by the quiet period, and
after the quiet period its timestamp is stale. A revoked drone has no record left under any identifier.

## ES–CS key management

The same two-message exchange runs with $ES_j$ as initiator and $CS_k$ as responder; $CS_k$ holds its own $X_{CS}$.
The resulting $SK_{ES,CS}$ protects partial blocks. The ECC/ECDSA blockchain phase of BAKMM-IoD is unchanged.
(For comparison, the ES–CS phase *as printed* in BAKMM-IoD fails; see the security notebook, check **E**.)

In [6]:
ra_c = F.RA(); cs = F.Responder(clock)            # cloud server = responder
cred_es = ra_c.register_drone(); cs.enroll(cred_es)
es_init = F.Initiator(cred_es, clock)             # ground station = initiator
for i in range(3):
    tr, a, b = F.run_session(es_init, cs, clock); clock.advance(30)
    print(f"ES-CS session {i+1}: SK_ES,CS agree = {a == b}, TIN on air = {tr['MSG1']['TID'].hex()[:12]}..")

ES-CS session 1: SK_ES,CS agree = True, TIN on air = dc2181e5d98c..
ES-CS session 2: SK_ES,CS agree = True, TIN on air = 519465679d27..
ES-CS session 3: SK_ES,CS agree = True, TIN on air = 8bd28cbf9a68..


**Output explanation.** Three ES–CS sessions succeed, and each uses a fresh pseudonym (TIN). Key evolution gives the
ES–CS link the same forward secrecy as the drone link.

## Sequence and architecture diagrams

Graphviz sources: `fslake_auth.dot`, `fslake_architecture.dot`. Graphviz is not installed on the authoring machine, so
the SVGs below were drawn by `proposed/code/diagrams.py` from the same content.

<img src="./fslake_auth.svg" width="900" alt="FSL-AKE-IoD message flow">

<img src="./fslake_architecture.svg" width="900" alt="FSL-AKE-IoD architecture">